In [10]:
import pandas as pd
from itertools import combinations
from typing import List, Set, Tuple, Dict
from mlxtend.frequent_patterns import apriori, association_rules

dataset = 'data_secretariado.csv'

In [3]:
class Apriori:
    """
    Implementación del algoritmo Apriori con caché de cálculos.
    """
    
    def __init__(self, soporte_min: float = 0.5, confianza_min: float = 0.5):
        """
        Inicializa el algoritmo Apriori optimizado.
        
        Parámetros:
        -----------
        soporte_min : float
            Umbral de soporte mínimo (0.0 a 1.0). Default: 0.5
        confianza_min : float
            Umbral de confianza mínimo (0.0 a 1.0). Default: 0.5
            
        Regresa:
        --------
        None
        """
        self.soporte_min = soporte_min
        self.confianza_min = confianza_min
        self.transacciones = []
        self.itemsets_frecuentes = {}
        self.reglas_asociacion = []
        
        # Caché de soportes
        self.cache_soporte = {}
        
        # Indexar transacciones
        self.num_transacciones = 0
    
    def carga_transacciones(self, datos: pd.DataFrame, columnas: List[str] = None) -> None:
        """
        Carga transacciones desde un DataFrame de pandas.
        
        Parámetros:
        -----------
        datos : pd.DataFrame
            DataFrame con los datos de transacciones
        columnas : List[str], optional
            Columnas a utilizar. Si es None, usa todas menos la primera (ID).
            
        Regresa:
        --------
        None
        """
        if columnas is None:
            columnas = datos.columns[1:]
        
        self.transacciones = []
        for idx, fila in datos.iterrows():
            transaccion = []
            for columna in columnas:
                if pd.notna(fila[columna]) and fila[columna] != "":
                    transaccion.append(f"{columna}={fila[columna]}")
            if transaccion:
                self.transacciones.append(frozenset(transaccion))
        
        # Guardar número de transacciones
        self.num_transacciones = len(self.transacciones)
        
        # Convertir a conjunto de conjuntos para búsqueda más rápida
        self.transacciones_set = [set(t) for t in self.transacciones]
        
        print(f"✓ {self.num_transacciones} transacciones cargadas")
    
    def calcula_soporte(self, itemset: frozenset) -> float:
        """
        Calcula el soporte de un itemset (con caché).
        
        Parámetros:
        -----------
        itemset : frozenset
            Conjunto de items para el cual calcular el soporte
            
        Regresa:
        --------
        float
            Valor de soporte entre 0.0 y 1.0
        """
        # Verificar caché primero
        if itemset in self.cache_soporte:
            return self.cache_soporte[itemset]
        
        # Contar transacciones que contienen el itemset
        contador = sum(1 for transaccion in self.transacciones_set 
                       if itemset.issubset(transaccion))
        soporte = contador / self.num_transacciones if self.num_transacciones > 0 else 0.0
        
        #Guardar en caché
        self.cache_soporte[itemset] = soporte
        
        return soporte
    
    def generar_candidatos_optimizado(self, itemsets: List[frozenset], k: int) -> List[frozenset]:
        """
        Genera candidatos de forma más eficiente (F-k-join).
        
        Parámetros:
        -----------
        itemsets : List[frozenset]
            Lista de itemsets frecuentes del nivel anterior
        k : int
            Tamaño del itemset a generar
            
        Regresa:
        --------
        List[frozenset]
            Lista de candidatos generados
        """
        if k == 2:
            # Para k=2, simplemente hacer combinaciones de items
            items = set()
            for itemset in itemsets:
                items.update(itemset)
            items = sorted(list(items))
            candidatos = [frozenset([items[i], items[j]]) 
                         for i in range(len(items)) 
                         for j in range(i + 1, len(items))]
        else:
            # Para k>2, unir itemsets que comparten k-2 items
            candidatos = set()
            itemsets_sorted = sorted([sorted(list(x)) for x in itemsets])
            
            for i in range(len(itemsets_sorted)):
                for j in range(i + 1, len(itemsets_sorted)):
                    # Si comparten los primeros k-2 elementos
                    if itemsets_sorted[i][:-1] == itemsets_sorted[j][:-1]:
                        union = frozenset(itemsets_sorted[i]) | frozenset(itemsets_sorted[j])
                        if len(union) == k:
                            candidatos.add(union)
            
            candidatos = list(candidatos)
        
        return candidatos
    
    def obtiene_itemsets_frecuentes(self, itemsets: List[frozenset]) -> List[frozenset]:
        """
        Filtra itemsets por el umbral de soporte mínimo.
        
        Parámetros:
        -----------
        itemsets : List[frozenset]
            Lista de itemsets candidatos
            
        Regresa:
        --------
        List[frozenset]
            Lista de itemsets que cumplen con soporte_min
        """
        frecuentes = []
        for itemset in itemsets:
            if self.calcula_soporte(itemset) >= self.soporte_min:
                frecuentes.append(itemset)
        
        return frecuentes
    
    def apriori(self) -> Dict[int, List[Tuple[frozenset, float]]]:
        """
        Ejecuta el algoritmo Apriori completo (optimizado).
        
        Parámetros:
        -----------
        (ninguno)
        
        Regresa:
        --------
        Dict[int, List[Tuple[frozenset, float]]]
            Diccionario de itemsets frecuentes por nivel
        """
        if not self.transacciones:
            raise ValueError("No hay transacciones cargadas. Usa carga_transacciones() primero.")
        
        print(f"\n Ejecutando Apriori (soporte_min={self.soporte_min})...")
        
        self.itemsets_frecuentes = {}
        
        # Genera itemsets de tamaño 1
        print(" Generando itemsets de tamaño 1...")
        items = set()
        for transaccion in self.transacciones:
            items.update(transaccion)
        
        candidatos_1 = [frozenset([item]) for item in items]
        frecuentes_1 = self.obtiene_itemsets_frecuentes(candidatos_1)
        
        if not frecuentes_1:
            print(" ✗ No se encontraron itemsets frecuentes")
            return self.itemsets_frecuentes
        
        self.itemsets_frecuentes[1] = [(itemset, self.calcula_soporte(itemset)) 
                                        for itemset in frecuentes_1]
        
        print(f" ✓ {len(frecuentes_1)} itemsets de tamaño 1")
        
        # Genera itemsets de tamaño k > 1
        k = 2
        frecuentes_actuales = frecuentes_1
        
        while frecuentes_actuales:
            print(f" Generando itemsets de tamaño {k}...")
            
            candidatos_k = self.generar_candidatos_optimizado(frecuentes_actuales, k)
            
            if not candidatos_k:
                break
            
            frecuentes_k = self.obtiene_itemsets_frecuentes(candidatos_k)
            
            if not frecuentes_k:
                print(f" ✓ {len(frecuentes_k)} itemsets de tamaño {k}")
                break
            
            self.itemsets_frecuentes[k] = [(itemset, self.calcula_soporte(itemset)) 
                                            for itemset in frecuentes_k]
            
            print(f" ✓ {len(frecuentes_k)} itemsets de tamaño {k}")
            
            frecuentes_actuales = frecuentes_k
            k += 1
        
        print(f"✓ Total de itemsets: {sum(len(v) for v in self.itemsets_frecuentes.values())}")
        
        return self.itemsets_frecuentes
    
    def calcula_confianza(self, antecedente: frozenset, consecuente: frozenset) -> float:
        """
        Calcula la confianza (con caché).
        
        Parámetros:
        -----------
        antecedente : frozenset
            Conjunto de items antecedentes
        consecuente : frozenset
            Conjunto de items consecuentes
            
        Regresa:
        --------
        float
            Valor de confianza entre 0.0 y 1.0
        """
        soporte_antecedente = self.calcula_soporte(antecedente)
        if soporte_antecedente == 0:
            return 0.0
        
        soporte_union = self.calcula_soporte(antecedente | consecuente)
        return soporte_union / soporte_antecedente
    
    def calcula_lift(self, antecedente: frozenset, consecuente: frozenset) -> float:
        """
        Calcula el lift (con caché).
        
        Parámetros:
        -----------
        antecedente : frozenset
            Conjunto de items antecedentes
        consecuente : frozenset
            Conjunto de items consecuentes
            
        Regresa:
        --------
        float
            Valor de lift
        """
        confianza = self.calcula_confianza(antecedente, consecuente)
        soporte_consecuente = self.calcula_soporte(consecuente)
        
        if soporte_consecuente == 0:
            return 0.0
        
        return confianza / soporte_consecuente
    
    def genera_reglas_asociacion(self) -> List[Dict]:
        """
        Genera reglas de asociación (optimizado).
        
        Parámetros:
        -----------
        (ninguno)
        
        Regresa:
        --------
        List[Dict]
            Lista de reglas de asociación
        """
        if not self.itemsets_frecuentes:
            raise ValueError("Primero debes ejecutar apriori()")
        
        print(f"\nGenerando reglas (confianza_min={self.confianza_min})...")
        
        self.reglas_asociacion = []
        contador_reglas = 0
        
        # Procesa itemsets de tamaño >= 2
        for k in sorted(self.itemsets_frecuentes.keys()):
            if k < 2:
                continue
            
            num_itemsets_k = len(self.itemsets_frecuentes[k])
            
            for idx, (itemset, soporte) in enumerate(self.itemsets_frecuentes[k]):
                if (idx + 1) % max(1, num_itemsets_k // 10) == 0:
                    print(f"  Procesando nivel {k}: {idx + 1}/{num_itemsets_k}")
                
                items = list(itemset)
                
                # OPTIMIZACIÓN: Solo generar particiones, no todas las combinaciones
                for r in range(1, len(items)):
                    for items_antecedente in combinations(items, r):
                        antecedente = frozenset(items_antecedente)
                        consecuente = itemset - antecedente
                        
                        confianza = self.calcula_confianza(antecedente, consecuente)
                        
                        if confianza >= self.confianza_min:
                            lift = self.calcula_lift(antecedente, consecuente)
                            
                            self.reglas_asociacion.append({
                                'antecedente': antecedente,
                                'consecuente': consecuente,
                                'soporte': soporte,
                                'confianza': confianza,
                                'lift': lift
                            })
                            
                            contador_reglas += 1
        
        print(f"{contador_reglas} reglas generadas")
        
        return self.reglas_asociacion
    
    def obtiene_reglas_dataframe(self) -> pd.DataFrame:
        """
        Convierte las reglas generadas a un DataFrame de pandas.
        
        Parámetros:
        -----------
        (ninguno)
        
        Regresa:
        --------
        pd.DataFrame
            DataFrame con las reglas
        """
        if not self.reglas_asociacion:
            raise ValueError("Primero debes ejecutar genera_reglas_asociacion()")
        
        df = pd.DataFrame({
            'antecedente': [regla['antecedente'] for regla in self.reglas_asociacion],
            'consecuente': [regla['consecuente'] for regla in self.reglas_asociacion],
            'soporte': [regla['soporte'] for regla in self.reglas_asociacion],
            'confianza': [regla['confianza'] for regla in self.reglas_asociacion],
            'lift': [regla['lift'] for regla in self.reglas_asociacion]
        })
        
        return df.sort_values('lift', ascending=False).reset_index(drop=True)
    
    def obtiene_itemsets_frecuentes_dataframe(self) -> pd.DataFrame:
        """
        Convierte los itemsets frecuentes a un DataFrame de pandas.
        
        Parámetros:
        -----------
        (ninguno)
        
        Regresa:
        --------
        pd.DataFrame
            DataFrame con los itemsets
        """
        datos = []
        for k, lista_itemsets in sorted(self.itemsets_frecuentes.items()):
            for itemset, soporte in lista_itemsets:
                datos.append({
                    'itemset': itemset,
                    'soporte': soporte,
                    'tamaño': k
                })
        
        df = pd.DataFrame(datos)
        return df.sort_values('soporte', ascending=False).reset_index(drop=True)

In [4]:
# Limpieza del dataset
datos = pd.read_csv(dataset)
# Como son columnas con valores de texto, no nos ayudan realmente a crear los valores de confianza et all.
columnas_a_remover = [
    'ID_VICTIMA',
    'CVE_ENT',
    'CVE_MUN',
    'FECHA_REGISTRO',
    'FECHA_NACIMIENTO',
    'FECHA_DESAPARICION',
    'ORIGEN_REPORTE'
]
datos_limpio = datos.drop(columns=columnas_a_remover, errors='ignore')
datos_limpio = datos_limpio[~(datos_limpio == 'CONFIDENCIAL').all(axis=1)]\

print(f"\nForma: {datos_limpio.shape}")
print(f"Columnas: {list(datos_limpio.columns)}")

print("Distribución de Valores x Columna")
for columna in datos_limpio.columns:
    valores_unicos = datos_limpio[columna].nunique()
    print(f"\n{columna}:")
    print(f"   - Valores únicos: {valores_unicos}")
    print(f"   - Top 5 valores más frecuentes:")
    print(datos_limpio[columna].value_counts().head(5))

print("Cálculo de Soporte Individual (Items de tamaño 1)")
soportes_items = {}

for columna in datos_limpio.columns:
    valores_unicos = datos_limpio[columna].dropna().unique()
    
    for valor in valores_unicos:
        if valor != 'CONFIDENCIAL':
            item = f"{columna}={valor}"
            num_ocurrencias = (datos_limpio[columna] == valor).sum()
            soporte = num_ocurrencias / len(datos_limpio)
            soportes_items[item] = soporte

# Mostrar items con mayor soporte
print("\nItems con mayor soporte")
items_ordenados = sorted(soportes_items.items(), key=lambda x: x[1], reverse=True)
for item, soporte in items_ordenados[:20]:
    print(f" {item}: {soporte:.4f}")

# Contar cuántos items tienen soporte >= X
for threshold in [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    count = sum(1 for s in soportes_items.values() if s >= threshold)
    print(f"Items con soporte >= {threshold}: {count}")


Forma: (133887, 4)
Columnas: ['SEXO', 'ESTATUS_VICTIMA', 'ENTIDAD', 'MUNICIPIO']
Distribución de Valores x Columna

SEXO:
   - Valores únicos: 4
   - Top 5 valores más frecuentes:
SEXO
HOMBRE           64666
CONFIDENCIAL     49149
MUJER            19716
INDETERMINADO      356
Name: count, dtype: int64

ESTATUS_VICTIMA:
   - Valores únicos: 3
   - Top 5 valores más frecuentes:
ESTATUS_VICTIMA
DESAPARECIDA     79881
CONFIDENCIAL     49149
NO LOCALIZADA     4857
Name: count, dtype: int64

ENTIDAD:
   - Valores únicos: 33
   - Top 5 valores más frecuentes:
ENTIDAD
JALISCO             14811
ESTADO DE MÉXICO    14344
TAMAULIPAS          13452
MICHOACÁN            7112
NUEVO LEÓN           7095
Name: count, dtype: int64

MUNICIPIO:
   - Valores únicos: 1554
   - Top 5 valores más frecuentes:
MUNICIPIO
CONFIDENCIAL    49149
SE DESCONOCE     5396
TIJUANA          2064
ATLAUTLA         1835
REYNOSA          1716
Name: count, dtype: int64
Cálculo de Soporte Individual (Items de tamaño 1)

Items 

In [21]:
# el valor de 0.05 es por los resultados anteriores.
apriori_manual = Apriori(soporte_min=0.05, confianza_min=0.3)
apriori_manual.carga_transacciones(datos_limpio, columnas=datos_limpio.columns)
itemsets_manual = apriori_manual.apriori()
df_itemsets_manual = apriori_manual.obtiene_itemsets_frecuentes_dataframe()
apriori_manual.genera_reglas_asociacion()
df_reglas_manual = apriori_manual.obtiene_reglas_dataframe()
# Debe de haber una mejor forma de mostrar la salida de esto.
print(df_reglas_manual)
df_reglas_manual.to_csv('datos_apriori_manual.csv', encoding='utf-8')

✓ 133887 transacciones cargadas

 Ejecutando Apriori (soporte_min=0.05)...
 Generando itemsets de tamaño 1...
 ✓ 13 itemsets de tamaño 1
 Generando itemsets de tamaño 2...
 ✓ 10 itemsets de tamaño 2
 Generando itemsets de tamaño 3...
 ✓ 4 itemsets de tamaño 3
 Generando itemsets de tamaño 4...
 ✓ 1 itemsets de tamaño 4
 Generando itemsets de tamaño 5...
✓ Total de itemsets: 28

Generando reglas (confianza_min=0.3)...
  Procesando nivel 2: 1/10
  Procesando nivel 2: 2/10
  Procesando nivel 2: 3/10
  Procesando nivel 2: 4/10
  Procesando nivel 2: 5/10
  Procesando nivel 2: 6/10
  Procesando nivel 2: 7/10
  Procesando nivel 2: 8/10
  Procesando nivel 2: 9/10
  Procesando nivel 2: 10/10
  Procesando nivel 3: 1/4
  Procesando nivel 3: 2/4
  Procesando nivel 3: 3/4
  Procesando nivel 3: 4/4
  Procesando nivel 4: 1/1
36 reglas generadas
                                          antecedente  \
0           frozenset({ESTATUS_VICTIMA=CONFIDENCIAL})   
1                      frozenset({SEXO=CONFI

In [6]:
# la verdad aqui si le pregunté a copilot que podrian significar eso de arriba:

# REGLA 0: ESTATUS_VICTIMA=CONFIDENCIAL => SEXO=CONFIDENCIAL
# Significa:
#   - Si el ESTATUS_VICTIMA es CONFIDENCIAL
#   - ENTONCES el SEXO también será CONFIDENCIAL
#   - Con 100% de confianza (1.0)
#   - Lift de 2.72 (correlación positiva muy fuerte)

# Interpretación:
# Cuando una víctima tiene estatus confidencial, 
# el sexo también tiende a ser confidencial.
# Esto es COHERENTE porque probablemente:
#   - Cuando no se reporta el estatus, tampoco se reporta el sexo
#   - Son ambos datos sensibles que van juntos

# Y, tiene cierta razón, pero siento que no termina de cuadrar. Al menos la primera regla claro. Y el mismo lift para todas reglas? 
# Quiero creer que tiene que ver por el valor de 0.05 que estamos agarrando. O tal vez sea probar con el mlxtend y ver que sale.

In [22]:
# por lo que encuentro, para pasarlo a algo que entienda el apriori de mlxtend, se debe convertir a un dataframe booleano/onehot.
# Osea, convertir cada valor en una columna booleana.

datos_para_mlxtend = datos_limpio.copy()

# En lugar de insertar columna por columna, crear una lista de columnas
columnas_onehot = {}

for columna in datos_para_mlxtend.columns:
    valores_unicos = datos_para_mlxtend[columna].dropna().unique()
    
    for valor in valores_unicos:
        nombre_col = f"{columna}={valor}"
        columnas_onehot[nombre_col] = (datos_para_mlxtend[columna] == valor).astype(bool)

datos_onehot = pd.DataFrame(columnas_onehot)
#print(datos_onehot.head())

itemsets_frequentes_mlxtend = apriori(datos_onehot, min_support=0.05, use_colnames=True)
reglas_mlxtend = association_rules(itemsets_frequentes_mlxtend, metric='confidence', min_threshold=0.3)
print(reglas_mlxtend[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
reglas_mlxtend.to_csv('datos_mlxtend.csv', encoding='utf-8')

                                          antecedents  \
0                      frozenset({SEXO=CONFIDENCIAL})   
1           frozenset({ESTATUS_VICTIMA=CONFIDENCIAL})   
2                        frozenset({ENTIDAD=JALISCO})   
3                      frozenset({SEXO=CONFIDENCIAL})   
4                 frozenset({MUNICIPIO=CONFIDENCIAL})   
5           frozenset({ESTATUS_VICTIMA=DESAPARECIDA})   
6                            frozenset({SEXO=HOMBRE})   
7                             frozenset({SEXO=MUJER})   
8                        frozenset({ENTIDAD=JALISCO})   
9           frozenset({ESTATUS_VICTIMA=CONFIDENCIAL})   
10                frozenset({MUNICIPIO=CONFIDENCIAL})   
11              frozenset({ENTIDAD=ESTADO DE MÉXICO})   
12                    frozenset({ENTIDAD=TAMAULIPAS})   
13                       frozenset({ENTIDAD=JALISCO})   
14    frozenset({ENTIDAD=JALISCO, SEXO=CONFIDENCIAL})   
15  frozenset({ENTIDAD=JALISCO, ESTATUS_VICTIMA=CO...   
16                       frozen